# 📊 Exploratory Data Analysis — RAVDESS Dataset

This notebook explores the RAVDESS (Ryerson Audio-Visual Database of Emotional Speech and Song) dataset used for Speech Emotion Recognition.

**Contents:**
1. Dataset overview: shape, class balance, gender split
2. Waveform plots: one example per emotion (8 subplots)
3. MFCC heatmaps: one per emotion
4. Mel spectrogram comparisons: Happy vs Sad vs Angry
5. Feature distribution: violin plots of ZCR and RMS per emotion
6. Correlation heatmap of top 40 MFCC features
7. t-SNE visualisation of extracted features coloured by emotion

In [ ]:
import sys, os
# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm.notebook import tqdm

from src import config
from src.data_loader import load_ravdess_dataset, get_class_distribution
from src.feature_extraction import extract_features

sns.set_theme(style='whitegrid', palette='viridis')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Dataset Overview

In [ ]:
df = load_ravdess_dataset()
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Unique emotions: {df["emotion_label"].nunique()}')
print(f'Unique actors: {df["actor"].nunique()}')
df.head(10)

In [ ]:
# Class balance
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Emotion counts
sns.countplot(data=df, x='emotion_label', order=sorted(df['emotion_label'].unique()),
              hue='emotion_label', palette='viridis', ax=axes[0], legend=False)
axes[0].set_title('Emotion Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

# Gender split
sns.countplot(data=df, x='gender', hue='gender', palette='Set2', ax=axes[1], legend=False)
axes[1].set_title('Gender Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 2. Waveform Plots — One per Emotion

In [ ]:
emotions = sorted(df['emotion_label'].unique())
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for idx, emotion in enumerate(emotions):
    ax = axes[idx // 4, idx % 4]
    sample = df[df['emotion_label'] == emotion].iloc[0]
    y, sr = librosa.load(sample['file_path'], sr=config.SAMPLE_RATE, duration=config.DURATION)
    librosa.display.waveshow(y, sr=sr, ax=ax, color=plt.cm.tab10(idx / len(emotions)))
    ax.set_title(f'{config.EMOTION_EMOJIS.get(emotion, "")} {emotion}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.suptitle('Waveforms — One Sample per Emotion', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. MFCC Heatmaps — One per Emotion

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for idx, emotion in enumerate(emotions):
    ax = axes[idx // 4, idx % 4]
    sample = df[df['emotion_label'] == emotion].iloc[0]
    y, sr = librosa.load(sample['file_path'], sr=config.SAMPLE_RATE, duration=config.DURATION)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=config.N_MFCC, hop_length=config.HOP_LENGTH)
    img = librosa.display.specshow(mfcc, x_axis='time', sr=sr, hop_length=config.HOP_LENGTH,
                                    ax=ax, cmap='magma')
    ax.set_title(f'{config.EMOTION_EMOJIS.get(emotion, "")} {emotion}', fontsize=12, fontweight='bold')
    ax.set_ylabel('MFCC')

plt.suptitle('MFCC Heatmaps — One Sample per Emotion', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Mel Spectrogram Comparisons: Happy vs Sad vs Angry

In [ ]:
compare_emotions = ['Happy', 'Sad', 'Angry']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, emotion in enumerate(compare_emotions):
    sample = df[df['emotion_label'] == emotion].iloc[0]
    y, sr = librosa.load(sample['file_path'], sr=config.SAMPLE_RATE, duration=config.DURATION)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=config.N_MELS, hop_length=config.HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, x_axis='time', y_axis='mel', sr=sr,
                                    hop_length=config.HOP_LENGTH, ax=axes[idx], cmap='inferno')
    axes[idx].set_title(f'{config.EMOTION_EMOJIS.get(emotion, "")} {emotion}', fontsize=14, fontweight='bold')
    fig.colorbar(img, ax=axes[idx], format='%+2.0f dB')

plt.suptitle('Mel Spectrogram Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Feature Distribution: Violin Plots of ZCR and RMS per Emotion

In [ ]:
# Compute ZCR and RMS for all files
zcr_list, rms_list, emo_list = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Computing ZCR & RMS'):
    try:
        y, sr = librosa.load(row['file_path'], sr=config.SAMPLE_RATE, duration=config.DURATION)
        zcr_list.append(np.mean(librosa.feature.zero_crossing_rate(y)))
        rms_list.append(np.mean(librosa.feature.rms(y=y)))
        emo_list.append(row['emotion_label'])
    except Exception:
        continue

feat_df = pd.DataFrame({'ZCR': zcr_list, 'RMS': rms_list, 'Emotion': emo_list})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.violinplot(data=feat_df, x='Emotion', y='ZCR', hue='Emotion',
               order=sorted(feat_df['Emotion'].unique()),
               palette='viridis', ax=axes[0], legend=False)
axes[0].set_title('Zero Crossing Rate by Emotion', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

sns.violinplot(data=feat_df, x='Emotion', y='RMS', hue='Emotion',
               order=sorted(feat_df['Emotion'].unique()),
               palette='viridis', ax=axes[1], legend=False)
axes[1].set_title('RMS Energy by Emotion', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap of Top 40 MFCC Features

In [ ]:
# Extract first 40 dims (MFCC mean) from the full 530-dim vector
mfcc_features = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting MFCC features'):
    feat = extract_features(row['file_path'])
    if feat is not None:
        mfcc_features.append(feat[:40])  # First 40 = MFCC means

mfcc_df = pd.DataFrame(mfcc_features, columns=[f'MFCC_{i+1}' for i in range(40)])

plt.figure(figsize=(16, 14))
corr = mfcc_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle mask
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=False, square=True, linewidths=0.3,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Top 40 MFCC Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. t-SNE Visualisation of Extracted Features

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Extract full 530-dim features for all files
all_features, all_labels = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building feature matrix'):
    feat = extract_features(row['file_path'])
    if feat is not None:
        all_features.append(feat)
        all_labels.append(row['emotion_label'])

X = np.array(all_features)
y_labels = np.array(all_labels)

# Standardise before t-SNE
X_scaled = StandardScaler().fit_transform(X)

# Run t-SNE (2D)
tsne = TSNE(n_components=2, random_state=config.RANDOM_STATE, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

tsne_df = pd.DataFrame({'Dim-1': X_tsne[:, 0], 'Dim-2': X_tsne[:, 1], 'Emotion': y_labels})

plt.figure(figsize=(12, 8))
sns.scatterplot(data=tsne_df, x='Dim-1', y='Dim-2', hue='Emotion',
                palette='tab10', alpha=0.7, s=60, edgecolor='white', linewidth=0.5)
plt.title('t-SNE Visualisation of 530-dim Features', fontsize=16, fontweight='bold')
plt.legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('EDA complete! ✅')